In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import date, timedelta
import requests
import json
import os
import time

bronze_df = spark.read.table(
    "clinical_trials.bronze_studies"
)

# print("Current Bronze rows:", bronze_df.count())

# bronze_dates_df = (
#     bronze_df
#     .select(
#         F.col(
#             "study.protocolSection.identificationModule.nctId"
#         ).alias("nct_id"),

#         F.to_date(
#             F.col(
#                 "study.protocolSection.statusModule.lastUpdatePostDateStruct.date"
#             )
#         ).alias("last_update_date")
#     )
# )

# # 같은 날짜 안에서도 우리가 실행한 이후 변경된 레코드가 있을 수 있기 때문
# # incremental pipeline에서는 일부 중복 수집을 허용하고, target write를 idempotent하게 만드는 방식이 더 안전한 경우가 많습니다.

# display(
#     bronze_dates_df
#     .orderBy(F.desc("last_update_date"))
#     .limit(20)
# )

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5516454050419190>, line 11
---> 11 t0 = perf_counter()
     13 df = spark.read.table(
     14     "clinical_trials.bronze_studies"
     15 )
     17 print(
     18     "read.table:",
     19     round(perf_counter() - t0, 2),
     20     "sec"
     21 )

NameError: name 'perf_counter' is not defined

In [0]:
import uuid

run_id = str(uuid.uuid4())
run_date = date.today().isoformat()

print("Run ID:", run_id)

Run ID: de321fed-100c-4695-b60b-76488205f1dd


In [0]:
# 현재 10,000건 데이터는 전체 registry의 full baseline이 아니므로
# sample의 MAX(last_update_date)를 실제 운영 watermark로 사용하지 않는다.
#
# Incremental ingestion과 MERGE 동작을 검증하기 위해
# 최근 3일을 명시적인 incremental window로 사용한다.

# incremental_from = (
#     date.today() - timedelta(days=3)
# ).isoformat()

# print(incremental_from)

In [0]:
# Watermark Table
#       ↓
# 지난 성공 날짜 읽기
#       ↓
# API query 자동 생성
# 기존 Cell 3의 하드 코딩 내역을 대체

PIPELINE_NAME = "clinical_trials_incremental"

watermark_row = (
    spark.read.table(
        "clinical_trials.pipeline_watermarks"
    )
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
    .first()
)

if watermark_row:
    last_watermark = watermark_row["watermark_date"]
    incremental_from = last_watermark.isoformat()
else:
    # 최초 incremental 실행 시 사용할 bootstrap window
    incremental_from = (
        date.today() - timedelta(days=3)
    ).isoformat()

print("Incremental from:", incremental_from)

update_query = (
    f"AREA[LastUpdatePostDate]"
    f"RANGE[{incremental_from},MAX]"
)

print("Query:", update_query)

Incremental from: 2026-09-25
Query: AREA[LastUpdatePostDate]RANGE[2026-09-25,MAX]


In [0]:
# Incremental API Query

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

update_query = (
    f"AREA[LastUpdatePostDate]"
    f"RANGE[{incremental_from},MAX]"
)

print(update_query)

AREA[LastUpdatePostDate]RANGE[2026-09-25,MAX]


In [0]:
params = {
    "query.term": update_query,
    "pageSize": 10,
    "format": "json"
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=30
)

response.raise_for_status()

test_data = response.json()

print("status:", response.status_code)
print("received:", len(test_data.get("studies", [])))
print("has next token:", bool(test_data.get("nextPageToken")))

status: 200
received: 10
has next token: True


In [0]:
for study in test_data.get("studies", [])[:5]:

    nct_id = (
        study
        .get("protocolSection", {})
        .get("identificationModule", {})
        .get("nctId")
    )

    last_update = (
        study
        .get("protocolSection", {})
        .get("statusModule", {})
        .get("lastUpdatePostDateStruct", {})
        .get("date")
    )

    print(nct_id, last_update)

NCT06459648 2026-09-25
NCT07196748 2026-09-25
NCT07842926 2026-09-25
NCT07626606 2026-09-25
NCT05160584 2026-09-25


In [0]:
# Incremental Raw 데이터를 페이지별로 저장

# 워터마크 적용 전
# raw_api/
# ├── ingest_date=2026-09-25/
# │   ├── page_0001.json
# │   └── ...
# │
# └── incremental/
#     └── run_date=2026-09-26/
#         ├── page_0001.json
#         └── ...

# 워터마크 업데이트 후 -> 같은 날 pipeline을 두 번 돌려도 첫 번째 Raw 파일을 덮어쓰지 않으려고 run id로 구분
# incremental/
# └── run_date=2026-09-26/
#     ├── run_id=abc.../
#     │   ├── page_0001.json
#     │   └── ...
#     └── run_id=def.../
#         └── ...

catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

run_date = date.today().isoformat()


# 동일 날짜에 pipeline이 여러 번 실행될 수 있으므로
# Raw 파일을 run_id 단위로 분리한다.
#
# 각 pipeline run은 자신의 raw directory만 읽도록 하여
# 이전 실행에서 남은 page 파일이 현재 batch에 섞이는 것을 방지한다.
incremental_raw_dir = (
    f"/Volumes/{catalog}/clinical_trials/raw_api/"
    f"incremental/"
    f"run_date={run_date}/"
    f"run_id={run_id}"
)

os.makedirs(
    incremental_raw_dir,
    exist_ok=True
)

print(incremental_raw_dir)

/Volumes/workspace/clinical_trials/raw_api/incremental/run_date=2026-09-26


In [0]:
# 실제 Incremental Pagination

PAGE_SIZE = 1000
MAX_PAGES = 20

next_page_token = None
total_studies = 0

for page_num in range(1, MAX_PAGES + 1):

    params = {
        "query.term": update_query,
        "pageSize": PAGE_SIZE,
        "format": "json"
    }

    if next_page_token:
        params["pageToken"] = next_page_token

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()
    studies = data.get("studies", [])

    file_path = (
        f"{incremental_raw_dir}/"
        f"page_{page_num:04d}.json"
    )

    with open(
        file_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False
        )

    total_studies += len(studies)

    print(
        f"page={page_num}, "
        f"received={len(studies)}, "
        f"total={total_studies}"
    )

    next_page_token = data.get(
        "nextPageToken"
    )

    if not next_page_token:
        print("Incremental ingestion completed.")
        break

    time.sleep(0.2)


page=1, received=1000, total=1000
page=2, received=366, total=1366
Incremental ingestion completed.


In [0]:
incremental_pages_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{incremental_raw_dir}/page_*.json")
)

print(
    "Incremental API pages:",
    incremental_pages_df.count()
)

incremental_studies_df = (
    incremental_pages_df
    .select(
        F.explode("studies").alias("study")
    )
)

print(
    "Incremental studies:",
    incremental_studies_df.count()
)

Incremental API pages: 4
Incremental studies: 2490


In [0]:
# Incremental batch에 metadata 추가

# bronze_df.printSchema()

incremental_bronze_df = (
    incremental_studies_df
    .withColumn(
        "source",
        F.lit("clinicaltrials.gov")
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "ingest_date",
        F.lit(run_date).cast("date")
    )
)

print(incremental_bronze_df.columns)

root
 |-- study: struct (nullable = true)
 |    |-- annotationSection: struct (nullable = true)
 |    |    |-- annotationModule: struct (nullable = true)
 |    |    |    |-- unpostedAnnotation: struct (nullable = true)
 |    |    |    |    |-- unpostedEvents: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- date: string (nullable = true)
 |    |    |    |    |    |    |-- dateUnknown: boolean (nullable = true)
 |    |    |    |    |    |    |-- type: string (nullable = true)
 |    |    |    |    |-- unpostedResponsibleParty: string (nullable = true)
 |    |-- derivedSection: struct (nullable = true)
 |    |    |-- conditionBrowseModule: struct (nullable = true)
 |    |    |    |-- ancestors: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |    |    |-- term: string (nullable = true)
 |    |    |  

In [0]:
# Merge 전 현재상태 기록

before_count = (
    spark.read.table(
        "clinical_trials.bronze_studies"
    ).count()
)

print("Before MERGE:", before_count)

Before MERGE: 13076


In [0]:
# Incremental 데이터 중 기존 ID 개수 확인

target_ids_df = (
    bronze_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id")
    )
)

incremental_ids_df = (
    incremental_bronze_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id")
    )
)

# 기존에도 존재하는 study
existing_updates = (
    incremental_ids_df
    .join(
        target_ids_df,
        on="nct_id",
        how="inner"
    )
    .count()
)

# 새로운 study
new_studies = (
    incremental_ids_df
    .join(
        target_ids_df,
        on="nct_id",
        how="left_anti"
    )
    .count()
)

print(
    "Existing IDs to potentially update:",
    existing_updates
)

print(
    "New IDs to insert:",
    new_studies
)

Existing IDs to potentially update: 2490
New IDs to insert: 0


In [0]:
# Incremental batch 내부 NCT ID 중복 확인
incremental_duplicate_ids = (
    incremental_bronze_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id")
    )
    .groupBy("nct_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate NCT IDs in incremental batch:",
    incremental_duplicate_ids.count()
)

display(incremental_duplicate_ids)

# NCT ID Null 확인
null_nct_ids = (
    incremental_bronze_df
    .filter(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).isNull()
    )
    .count()
)

print("Null NCT IDs:", null_nct_ids)



Duplicate NCT IDs in incremental batch: 494


nct_id,count
NCT05933018,2
NCT07842679,2
NCT07840794,2
NCT07222566,2
NCT06936527,2
NCT04989803,2
NCT07840716,2
NCT06106295,2
NCT07423780,2
NCT07843134,2


Null NCT IDs: 0


In [0]:
# Incremental JSON에서 Spark가 추론한 nested study schema는
# 기존 Bronze Delta Table의 schema와 다를 수 있다.

# study struct를 JSON 문자열로 직렬화한 뒤,
# 기존 Bronze의 target_study_schema를 사용해 다시 파싱함으로써
# incremental source의 nested schema를 target schema에 맞춘다.

# Target에 없는 source field는 제외되고,
# Target에는 있으나 source에 없는 field는 null로 채워진다.

# Raw JSON은 별도로 보존하므로 향후 source schema 변경이 확인되면
# schema evolution 후 재처리할 수 있다.

target_bronze_df = spark.read.table(
    "clinical_trials.bronze_studies"
)

# 기존 Delta Table의 study 구조를 꺼내기 -> 앞으로 study는 이 설계도를 따라야 한다
target_study_schema = (
    target_bronze_df.schema["study"].dataType
)

# Incremental 데이터의 study 구조를 기존 Bronze Delta Table이 사용하는 study 스키마에 강제로 맞추는 작업
# 복잡한 nested schema는 직접 .cast(target_study_schema)하면 호환되지 않는 경우가 많다.
incremental_aligned_df = (
    incremental_bronze_df
    .select(
        # F.from_json(JSON문자열, target_study_schema) : 이 JSON 문자열을 내가 지정한 schema대로 다시 읽어라. -> Target schema에 없는 값은 버려지고, Target에는 있는데 JSON에 없는 값은 null로 처리됨
        # 따라서 ClinicalTrials.gov에 정말 중요한 새 필드가 추가되어도 Target schema에 없다면 그 필드는 Bronze Delta에 들어가지 않습니다. -> schema enforcement에는 좋지만 schema evolution을 자동 처리하는 코드는 아닙니다.
        # 이에 Raw JSON을 따로 보존했음 (cell 6) -> 나중에 새 필드가 추가되어 Target schema를 수정 한뒤 Raw에서 다시 processing 할 수 있음 -> Raw layer를 보존하는 중요한 이유
        F.from_json(
            # F.col("study") : 현재 DataFrame의 study 컬럼을 가져와라
            # F.to_json(F.col("study")) : Spark의 struct를 JSON 문자열로 한번 풀어버리는 것 -> 기존 struct 자체에는 이미 Spark가 추론한 schema가 붙어 있기 때문입니다. 한번 JSON 문자열로 바꾸면 그 struct schema를 사실상 벗어던질 수 있습니다.
            F.to_json(F.col("study")),
            target_study_schema
        ).alias("study"),

        # 해당 컬럼들은 nested 구조가 아니므로 굳이 변환할 필요가 없음
        F.col("source"),
        F.col("ingested_at"),
        F.col("ingest_date")
    )
)

# 컬럼명과 타입뿐 아니라 nullable 같은 속성까지 비교
# F.lit("clinicaltrials.gov")로 만든 값은 절대 null이 아니기 때문에 Spark가 nullable=False로 판단할 수 있습니다. 반면 Delta Table을 다시 읽으면 해당 컬럼이 nullable=True로 표현되는 경우가 있습니다.
print(target_bronze_df.schema == incremental_aligned_df.schema)

False


In [0]:
source_with_key_df = (
    incremental_aligned_df
    .withColumn(
        "nct_id",
        F.col(
            "study.protocolSection.identificationModule.nctId"
        )
    )
    .withColumn(
        "study_hash",
        F.sha2(
            F.to_json(F.col("study")),
            256
        )
    )
)

duplicate_profile_df = (
    source_with_key_df
    .groupBy("nct_id")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("study_hash").alias("distinct_versions")
    )
    .filter(F.col("row_count") > 1)
)

display(duplicate_profile_df)

print(
    "Duplicated NCT IDs:",
    duplicate_profile_df.count()
)

deduped_source_df = (
    source_with_key_df
    .dropDuplicates([
        "nct_id",
        "study_hash"
    ])
)

conflicting_ids_df = (
    deduped_source_df
    .groupBy("nct_id")
    .agg(
        F.countDistinct("study_hash")
        .alias("distinct_versions")
    )
    .filter(
        F.col("distinct_versions") > 1
    )
)

conflicting_count = conflicting_ids_df.count()

print(
    "Conflicting duplicate NCT IDs:",
    conflicting_count
)

display(conflicting_ids_df)

nct_id,row_count,distinct_versions


Duplicated NCT IDs: 0
Conflicting duplicate NCT IDs: 0


nct_id,distinct_versions


In [0]:
# Spark StructType의 전체 equality 비교는 data type뿐 아니라
# nullable 등의 schema metadata도 포함한다.
# 따라서 전체 schema equality가 False이더라도,
# MERGE에 필요한 컬럼명과 실제 data type이 동일한지 별도로 확인한다.

# Top-level 타입 비교
for target_field, source_field in zip(
    target_bronze_df.schema.fields,
    incremental_aligned_df.schema.fields
):
    print(
        target_field.name,
        "| same name:",
        target_field.name == source_field.name,
        "| same datatype:",
        target_field.dataType == source_field.dataType,
        "| target nullable:",
        target_field.nullable,
        "| source nullable:",
        source_field.nullable
    )

# Study schema 확인
print(
    "Study schema same:",
    target_bronze_df.schema["study"].dataType
    ==
    incremental_aligned_df.schema["study"].dataType
)

# 나머지 top-level data atype 확인
for col_name in [
    "source",
    "ingested_at",
    "ingest_date"
]:
    target_type = (
        target_bronze_df.schema[col_name].dataType
    )

    source_type = (
        incremental_aligned_df.schema[col_name].dataType
    )

    print(
        col_name,
        target_type,
        source_type,
        target_type == source_type
    )

study | same name: True | same datatype: True | target nullable: True | source nullable: True
source | same name: True | same datatype: True | target nullable: True | source nullable: False
ingested_at | same name: True | same datatype: True | target nullable: True | source nullable: False
ingest_date | same name: True | same datatype: True | target nullable: True | source nullable: True
Study schema same: True
source StringType() StringType() True
ingested_at TimestampType() TimestampType() True
ingest_date DateType() DateType() True


In [0]:
# Merge 전 상태 기록

before_count = target_bronze_df.count()

print("Before MERGE:", before_count)
print("Expected inserts:", new_studies)
print("Expected matches:", existing_updates)

Before MERGE: 10000
Expected inserts: 3076
Expected matches: 48


In [0]:
# # Delta MERGE

# from delta.tables import DeltaTable

# target_table = DeltaTable.forName(
#     spark,
#     "clinical_trials.bronze_studies"
# )

# incremental_merge_df = (
#     deduped_source_df
#     .drop(
#         "nct_id",
#         "study_hash"
#     )
# )

# (
#     target_table.alias("target")
#     .merge(
#         incremental_merge_df.alias("source"),
#         """
#         target.study.protocolSection.identificationModule.nctId
#         =
#         source.study.protocolSection.identificationModule.nctId
#         """
#     )
#     .whenMatchedUpdateAll()
#     .whenNotMatchedInsertAll()
#     .execute()
# )

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# # 3,124건을 incremental batch로 처리했지만 table은 3,076건만 증가
# after_count = (
#     spark.read.table(
#         "clinical_trials.bronze_studies"
#     ).count()
# )

# print("Before:", before_count)
# print("After :", after_count)
# print("Increase:", after_count - before_count)

Before: 10000
After : 13076
Increase: 3076


In [0]:
%sql
-- Delta History 확인

DESCRIBE HISTORY clinical_trials.bronze_studies;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-09-26T06:40:48.000Z,71077798721055,warwarsn@gmail.com,MERGE,"Map(predicate -> [""(study#11793.protocolSection.identificationModule.nctId = study#11705.protocolSection.identificationModule.nctId)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3094780928041764),cae6a139-041d-4185-b998-9c771ebef955,0926-061343-bmq1pdre-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 13882329, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 4, numTargetRowsMatchedUpdated -> 48, executionTimeMs -> 12058, materializeSourceTimeMs -> 1384, numTargetRowsInserted -> 3076, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3038, numTargetRowsUpdated -> 48, numOutputRows -> 3124, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3124, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 7482)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-25T16:10:15.000Z,71077798721055,warwarsn@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(363666689905209),b8d182b1-49f2-425b-b629-f9b04081420c,0925-160029-pga7lm0n-v2n,0,WriteSerializable,false,"Map(numFiles -> 4, numRemovedFiles -> 1, numRemovedBytes -> 3363299, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000, numOutputBytes -> 31897416)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-25T08:26:57.000Z,71077798721055,warwarsn@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(26051292554930),fe5f2532-449b-4dc0-a586-550e4c7c8e02,0925-082200-iij2g1fz-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000, numOutputBytes -> 3363299)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


In [0]:
# # operationMetrics : Databricks가 실제로 처리한 row 수 정보를 기록

# merge_history_df = (
#     spark.sql("""
#         DESCRIBE HISTORY
#         clinical_trials.bronze_studies
#     """)
#     .select(
#         "version",
#         "timestamp",
#         "operation",
#         "operationMetrics"
#     )
# )

# display(merge_history_df.limit(5))

version,timestamp,operation,operationMetrics
2,2026-09-26T06:40:48.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 13882329, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 4, numTargetRowsMatchedUpdated -> 48, executionTimeMs -> 12058, materializeSourceTimeMs -> 1384, numTargetRowsInserted -> 3076, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3038, numTargetRowsUpdated -> 48, numOutputRows -> 3124, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3124, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 7482)"
1,2026-09-25T16:10:15.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 4, numRemovedFiles -> 1, numRemovedBytes -> 3363299, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000, numOutputBytes -> 31897416)"
0,2026-09-25T08:26:57.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000, numOutputBytes -> 3363299)"


In [0]:
# # 실제 기존 ID 하나가 들어있는지 확인

# existing_example = (
#     incremental_ids_df
#     .join(
#         target_ids_df,
#         on="nct_id",
#         how="inner"
#     )
#     .first()
# )

# print(existing_example["nct_id"])

# example_nct_id = existing_example["nct_id"]

# display(
#     spark.read.table(
#         "clinical_trials.bronze_studies"
#     )
#     .filter(
#         F.col(
#             "study.protocolSection.identificationModule.nctId"
#         ) == example_nct_id
#     )
# )

NCT06701214


study source ingested_at ingest_date List(null, List(null, List(List(List(D013048, Specimen Handling), List(D019411, Clinical Laboratory Techniques), List(D019937, Diagnostic Techniques and Procedures), List(D003933, Diagnosis), List(D011677, Punctures), List(D013514, Surgical Procedures, Operative), List(D008919, Investigative Techniques)), List(List(D001800, Blood Specimen Collection))), List(null, null, 2026-09-25)), null, false, List(List(null, List(List(null, Blood will be taken from participants already undergoing blood collection, Blood collection, null, OTHER), List(null, Undergo saliva collection, Saliva collection, null, OTHER), List(null, Bone marrow will be taken from participants already undergoing bone marrow collection, Bone marrow collection, null, OTHER))), List(List(Clonal Hematopoiesis), null), List(List(List(cip@vumc.org, Vanderbilt-Ingram Services for Timely Access, 800-811-8480, null, CONTACT)), List(List(Nashville, List(List(cip@vumc.org, Vanderbilt-Ingram Service Services for Timely Access, 800-811-8480, null, CONTACT), List(null, Alexander Bick, MD, PhD, null, null, PRINCIPAL_INVESTIGATOR)), United States, Vanderbilt University/Ingram Cancer Center, List(36.16589, -86.78444), Tennessee, RECRUITING, 37203)), List(List(Vanderbilt University/Ingram Cancer Center, Alexander Bick, MD, PhD, PRINCIPAL_INVESTIGATOR))), List(This study will investigate the association between clonal hematopoiesis and other conditions. Clonal hematopoiesis (CH) refers to the mutations in a person's stem cells which commonly affect people as they get older. These mutations have notably been linked to increased risk of certain cancers as well as increased risk of heart disease., Objectives:

* To develop the CHIVE Registry with CH patients or patients with elevated risk for CH in order to better characterize associated clinical outcomes and co-morbidities in these populations
* To improve understanding of cardiovascular phenotypes and risk of patients with known Clonal Hematopoiesis of indeterminate potential (CHIP) or Clonal Cytopenias of undetermined significance (CCUS)
* To investigate prognostic significance of CH mutations for the development hematologic malignancies
* To generate a CHIVE Biorepository to acquire patient samples to study the natural history of CH, and to make available to CH investigators.), List(null, List(null, null, null, null, COHORT, null, PROSPECTIVE), List(800, ESTIMATED), null, null, true, null, OBSERVATIONAL, 10 Years), List(Inclusion Criteria:

* Patient greater than or equal to 18 years old at time of consent
* Able to provide informed consent
* Idiopathic cytopenia (ICUS) or idiopathic cytoses (elevated blood counts without disease or explanation); clonal cytopenia of undetermined significance (CCUS), clonal hematopoiesis of indeterminate potential (CHIP) or individuals at higher risk for clonal hematopoiesis (ex. patients with known diagnosis of solid tumors or cardiovascular disease)

Exclusion Criteria:

* Unable to provide consent
* Diagnosis of active hematologic malignancy. For example, a diagnosis of CMML, AML, MDS, MPN; History of hematologic malignancy is NOT exclusionary if in complete remission (e.g. previous myeloma or lymphoma), null, null, false, null, 18 Years, NON_PROBABILITY_SAMPLE, ALL, List(ADULT, OLDER_ADULT), Adults \>/= 18 years old with a mutation known to cause clonal hematopoiesis or at a higher risk of having a clonal hematopoiesis mutation.), List(CHIVE, The Clonal Hematopoiesis & Inflammation in Vasculature Registry and Biorepository, NCT06701214, null, Discovering Outcomes in Clonal Hematopoiesis: The Clonal Hematopoiesis and Inflammation in VasculaturE (CHIVE) Registry and Biorepository, List(VICCHEM20123, null, null), List(OTHER, Vanderbilt-Ingram Cancer Center), List(List(NCI, Clinical Trials Reporting Program, NCI-2021-00704, null, REGISTRY))), List(null, null, null, UNDECIDED, null, null), List(null, List(List(Aggregate data will be utilized to better understand t

In [0]:
# # 동일한 incremental batch를 재실행해도 target row 수가 증가하지 않는지 검증한다.
# # NCT ID를 MERGE key로 사용하므로 기존 record는 UPDATE되고,
# # 이미 INSERT된 record 역시 두 번째 실행에서는 MATCHED 처리된다.
# # 이를 통해 pipeline의 idempotency를 확인한다.

# # 멱등성 = 최종 상태가 같다
# # 비효율 없음 = 같은 row를 다시 쓰지 않는다

# # MERGE 전 기록
# before_second_merge = (
#     spark.read.table(
#         "clinical_trials.bronze_studies"
#     ).count()
# )

# print("Before second MERGE:", before_second_merge)

# # 일한 incremental_aligned_df로 MERGE를 한 번 더 실행
# target_table = DeltaTable.forName(
#     spark,
#     "clinical_trials.bronze_studies"
# )

# (
#     target_table.alias("target")
#     .merge(
#         incremental_aligned_df.alias("source"),
#         """
#         target.study.protocolSection.identificationModule.nctId
#         =
#         source.study.protocolSection.identificationModule.nctId
#         """
#     )
#     .whenMatchedUpdateAll()
#     .whenNotMatchedInsertAll()
#     .execute()
# )

# after_second_merge = (
#     spark.read.table(
#         "clinical_trials.bronze_studies"
#     ).count()
# )

# print("Before:", before_second_merge)
# print("After :", after_second_merge)
# print(
#     "Row increase:",
#     after_second_merge - before_second_merge
# )

Before second MERGE: 13076
Before: 13076
After : 13076
Row increase: 0


In [0]:
%sql
-- .whenMatchedUpdateAll() : 실제로 내용이 같더라도 matched row 전체에 UPDATE를 수행할 수 있습니다.
-- 현재 멱등성은 확보하나 불필요한 update를 피하는 최적화는 되지 않은 상태

DESCRIBE HISTORY clinical_trials.bronze_studies;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-09-26T06:47:59.000Z,71077798721055,warwarsn@gmail.com,MERGE,"Map(predicate -> [""(study#15165.protocolSection.identificationModule.nctId = study#11705.protocolSection.identificationModule.nctId)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3094780928041764),a000d67f-182c-4a8b-ad9e-fa8a27052cd1,0926-061343-bmq1pdre-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 13882329, numTargetBytesRemoved -> 13882329, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 3124, executionTimeMs -> 6294, materializeSourceTimeMs -> 849, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2227, numTargetRowsUpdated -> 3124, numOutputRows -> 3124, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3124, numTargetFilesRemoved -> 4, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3107)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
2,2026-09-26T06:40:48.000Z,71077798721055,warwarsn@gmail.com,MERGE,"Map(predicate -> [""(study#11793.protocolSection.identificationModule.nctId = study#11705.protocolSection.identificationModule.nctId)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3094780928041764),cae6a139-041d-4185-b998-9c771ebef955,0926-061343-bmq1pdre-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 13882329, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 4, numTargetRowsMatchedUpdated -> 48, executionTimeMs -> 12058, materializeSourceTimeMs -> 1384, numTargetRowsInserted -> 3076, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3038, numTargetRowsUpdated -> 48, numOutputRows -> 3124, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3124, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 7482)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-25T16:10:15.000Z,71077798721055,warwarsn@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(363666689905209),b8d182b1-49f2-425b-b629-f9b04081420c,0925-160029-pga7lm0n-v2n,0,WriteSerializable,false,"Map(numFiles -> 4, numRemovedFiles -> 1, numRemovedBytes -> 3363299, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000, numOutputBytes -> 31897416)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-25T08:26:57.000Z,71077798721055,warwarsn@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(26051292554930),fe5f2532-449b-4dc0-a586-550e4c7c8e02,0925-082200-iij2g1fz-v2

In [0]:
# 조건부 MERGE
# Target study를 JSON으로 바꿔 hash를 만들고, Source study도 JSON으로 바꿔 hash를 만든 뒤, 둘이 다를 때만 UPDATE한다.

# 최초 MERGE는 whenMatchedUpdateAll을 사용해 동일 batch 재실행 시에도
# 모든 matched record가 UPDATE되는 문제가 있었다.
#
# 최종 상태는 동일하므로 idempotency는 만족하지만,
# 실제 변경이 없는 record까지 rewrite하는 것은 비효율적이다.
#
# 따라서 target/source의 study JSON을 SHA-256 hash로 비교하고,
# 실제 study 내용이 변경된 경우에만 UPDATE하도록 MERGE 조건을 개선한다.
#
# 동일 incremental batch 재실행 시:
# INSERT = 0 / UPDATE = 0 을 기대한다.

target_table = DeltaTable.forName(
    spark,
    "clinical_trials.bronze_studies"
)

incremental_merge_df = (
    deduped_source_df
    .drop(
        "nct_id",
        "study_hash"
    )
)

(
    target_table.alias("target")
    .merge(
        incremental_merge_df.alias("source"),
        """
        target.study.protocolSection.identificationModule.nctId
        =
        source.study.protocolSection.identificationModule.nctId
        """
    )
    .whenMatchedUpdate(
        condition="""
            sha2(to_json(target.study), 256)
            <>
            sha2(to_json(source.study), 256)
        """,
        set={
            "study": "source.study",
            "source": "source.source",
            "ingested_at": "source.ingested_at",
            "ingest_date": "source.ingest_date"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)


before_third_merge = (
    spark.read.table(
        "clinical_trials.bronze_studies"
    ).count()
)

after_third_merge = (
    spark.read.table(
        "clinical_trials.bronze_studies"
    ).count()
)

print("Before:", before_third_merge)
print("After :", after_third_merge)
print(
    "Increase:",
    after_third_merge - before_third_merge
)

Before: 13076
After : 13076
Increase: 0


In [0]:
%sql

DESCRIBE HISTORY clinical_trials.bronze_studies;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-09-26T06:52:16.000Z,71077798721055,warwarsn@gmail.com,MERGE,"Map(predicate -> [""(study#17734.protocolSection.identificationModule.nctId = study#11705.protocolSection.identificationModule.nctId)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (sha2(cast(to_json(study#17734, Some(Etc/UTC)) as binary), 256) = sha2(cast(to_json(study#11705, Some(Etc/UTC)) as binary), 256))"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3094780928041764),4ace901e-58d9-4e65-a559-e0a30838f131,0926-061343-bmq1pdre-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 8450, materializeSourceTimeMs -> 819, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3321, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3124, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4254)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
3,2026-09-26T06:47:59.000Z,71077798721055,warwarsn@gmail.com,MERGE,"Map(predicate -> [""(study#15165.protocolSection.identificationModule.nctId = study#11705.protocolSection.identificationModule.nctId)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3094780928041764),a000d67f-182c-4a8b-ad9e-fa8a27052cd1,0926-061343-bmq1pdre-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 13882329, numTargetBytesRemoved -> 13882329, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 3124, executionTimeMs -> 6294, materializeSourceTimeMs -> 849, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2227, numTargetRowsUpdated -> 3124, numOutputRows -> 3124, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3124, numTargetFilesRemoved -> 4, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3107)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
2,2026-09-26T06:40:48.000Z,71077798721055,warwarsn@gmail.com,MERGE,"Map(predicate -> [""(study#11793.protocolSection.identificationModule.nctId = study#11705.protocolSection.identificationModule.nctId)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3094780928041764),cae6a139-041d-4185-b998-9c771ebef955,0926-061343-bmq1pdre-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 13882329, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 4, numTargetRowsMatchedUpdated -> 48, executionTimeMs -> 12058, materializeSourceTimeMs -> 1384, numTargetRowsInserted -> 3076, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3038, numTargetRowsUpdated -> 48, numOutputRows -> 3124, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3124, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeM

In [0]:
%sql
-- Watermark 관리용 Delta Table 생성

CREATE TABLE IF NOT EXISTS clinical_trials.pipeline_watermarks (
    pipeline_name STRING,
    watermark_date DATE,
    last_run_id STRING,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:
# Watermark
# MERGE 성공 후에만 Watermark를 갱신 -> 마지막으로 시도한 시점이 아니라 마지막으로 성공한 시점

processed_batch_df = (
    incremental_aligned_df
    .select(
        F.to_date(
            F.col(
                "study.protocolSection.statusModule.lastUpdatePostDateStruct.date"
            )
        ).alias("last_update_date")
    )
)

processed_watermark = (
    processed_batch_df
    .agg(
        F.max("last_update_date").alias("watermark")
    )
    .first()["watermark"]
)

print("Processed watermark:", processed_watermark)

Processed watermark: 2026-09-25


In [0]:
# 성공 결과를 Watermark Table에 기록

# watermark update용 DataFrame
watermark_update_df = (
    spark.createDataFrame(
        [
            (
                "clinical_trials_incremental",
                processed_watermark,
                run_id
            )
        ],
        [
            "pipeline_name",
            "watermark_date",
            "last_run_id"
        ]
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)


# Watermark Table MERGE
watermark_table = DeltaTable.forName(
    spark,
    "clinical_trials.pipeline_watermarks"
)

(
    watermark_table.alias("target")
    .merge(
        watermark_update_df.alias("source"),
        "target.pipeline_name = source.pipeline_name"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

display(
    spark.read.table(
        "clinical_trials.pipeline_watermarks"
    )
)

pipeline_name,watermark_date,last_run_id,updated_at
clinical_trials_incremental,2026-09-25,f0884bcb-6017-48bc-a0c2-1c5024b5447f,2026-09-26T07:01:50.554Z
